In [0]:
%run "/Workspace/Users/rahulpatel@cyntexa.com/DataEngineering-Project/de_project/src/includes"

In [0]:
# to install the faker and restart the python on workspace.
 
%pip install faker
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("catalog" , "de_dev")
print(dbutils.widgets.get("catalog"))

In [0]:
from faker import Faker
import pandas as pd
import random

from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
fake = Faker()

# save
catalog = dbutils.widgets.get("catalog")
OUTPUT_PATH = f"/Volumes/{catalog}/bronze/raw"

In [0]:
suppliers = []

for i in range(1, 501):
    suppliers.append({
        "supplier_id": i,
        "supplier_name": fake.company(),
        "contact_email": None if random.random() < 0.05 else fake.company_email(),
        "country": fake.country()
    })

supplier_df = pd.DataFrame(suppliers)

supplier_spark = spark.createDataFrame(supplier_df)

(supplier_spark
 .coalesce(1)
 .write
 .mode("overwrite")
 .option("header", True)
 .csv(f"{OUTPUT_PATH}/suppliers"))

In [0]:
customers = []

for i in range(1, 1000):
    customers.append({
        "customer_id": i,
        "name": fake.name(),
        "email": None if random.random() < 0.05 else fake.email(),
        "city": fake.city(),
        "state": fake.state(),
        "signup_date": fake.date_between(start_date="-5y", end_date="today"),
        "phone": fake.phone_number(),
        "customer_segment": random.choice(
            ["Premium", "Standard", "Basic"]
        )
    })

customer_df = pd.DataFrame(customers)

duplicates = customer_df.sample(100)
customer_df = pd.concat([customer_df, duplicates], ignore_index=True)

customer_spark = spark.createDataFrame(customer_df)

(customer_spark
 .coalesce(1)
 .write
 .mode("overwrite")
 .option("header", True)
 .csv(f"{OUTPUT_PATH}/customers"))

In [0]:
categories = [
    "Electronics",
    "Fashion",
    "Home",
    "Sports",
    "Books",
    "Beauty"
]

products = []

for i in range(100, 2000):
    products.append({
        "product_id": i,
        "product_name": fake.catch_phrase(),
        "category": random.choice(categories),
        "price": None if random.random() < 0.05 else round(random.uniform(10, 10000), 2),
        "supplier_id": random.randint(1, 500),
        "sku": fake.bothify(text="SKU-####")
    })

product_df = pd.DataFrame(products)

product_df = pd.concat(
    [product_df, product_df.sample(50)],
    ignore_index=True
)

product_spark = spark.createDataFrame(product_df)

(product_spark
 .coalesce(1)
 .write
 .mode("overwrite")
 .option("header", True)
 .csv(f"{OUTPUT_PATH}/products"))

In [0]:
sales = []

for i in range(1, 200001):
    sales.append({
        "sale_id": i,
        "customer_id": random.randint(1, 20500),
        "product_id": random.randint(1, 5100),
        "quantity": None if random.random() < 0.03 else random.randint(1, 20),
        "sale_amount": round(random.uniform(100, 50000), 2),
        "discount": round(random.uniform(0, 30), 2),
        "sale_date": fake.date_between(start_date="-3y", end_date="today"),
        "region": random.choice(["North", "South", "East", "West"]),
        "payment_method": random.choice(
            ["Credit Card", "UPI", "Cash", "Net Banking"]
        ),
        "order_status": random.choice(
            ["Completed", "Pending", "Cancelled"]
        )
    })

sales_df = pd.DataFrame(sales)

sales_spark = spark.createDataFrame(sales_df)

(sales_spark
 .coalesce(1)
 .write
 .mode("overwrite")
 .option("header", True)
 .csv(f"{OUTPUT_PATH}/sales"))